## Exercise 4: Data Warehouse Querying and Basic Geospatial Operations

Skills: 
* Query data warehouse table
* Use dictionary to map values

References: 
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-basics.html
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-intro.html
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-intermediate.html
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [ ]:
import geopandas as gpd
import pandas as pd
import os

#os.environ["CALITP_BQ_MAX_BYTES"] = str(100_000_000_000)
pd.set_option("display.max_rows", 20)

from calitp.tables import tbl
from calitp import query_sql
from siuba import *
import shared_utils

## Query a table, turn it into a gdf
For two operators, `ITP_ID == 246` (Caltrain), `ITP_ID == 279` (BART) and `ITP_ID == 343` (Merced the Bus), you will query the warehouse table.

* Query `views.gtfs_schedule_dim_stops`
* Filter to the ITP_ID of interest
* Select these columns: `calitp_itp_id`, `stop_id`, `stop_lat`, `stop_lon`, `stop_name`
* Return as a dataframe using `collect()`
* Turn the point data into geometry with `geopandas`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html)

## Amanda 1/11:  added distinct() to query. Help.
* Tried to do select distinct (_.itp_id...stop_name) but didn't work

In [ ]:
ITP_ID = [246, 279, 343]
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select (_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

## Use a dictionary to map values

* Create a new column called `operator` where `itp_id` is associated with its operator name.
* First, write a function to do it.
* Then, use a dictionary to do it (create new column called `agency`).
* Double check that `operator` and `agency` show the same values. Use `assert` to check.
    * `assert df.operator == df.agency`
* Hint: https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html

In [ ]:
#previewing
stops.head(2)

In [ ]:
#previewing
stops.shape

In [ ]:
stops.dtypes

#### Function Method

In [ ]:
def operator_names(x):
    if x.itp_id == 246:
        return "Caltrain"
    elif x.itp_id == 279:
        return "BART"
    else:
        return "Merced the Bus"

In [ ]:
stops["operator"] = stops.apply(lambda x: operator_names(x), axis=1) 

#### Dictionary Method

In [ ]:
agency_names = {279: 'BART', 246:'Caltrain', 343: 'Merced the Bus'}

In [ ]:
stops["agency"] = stops.itp_id.map(agency_names)

In [ ]:
stops.head(2)

#### Checking that both columns are the same 

In [ ]:
assert (stops.operator == stops.agency).all()

## Turn lat/lon into point geometry

There is a [function in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py#L170-L192) that does it. Show the steps within the function (the long way), and also create the `geometry` column using `shared_utils`.

In [ ]:
#showing steps
stops2 = stops.assign(
      geometry = gpd.points_from_xy(stops['stop_lon'], stops['stop_lat'], 
      crs = 'WGS84'))

gdf = gpd.GeoDataFrame(stops2)

In [ ]:
gdf.head(2)

In [ ]:
#using the function
shared_utils.geography_utils.create_point_geometry(stops, 'stop_lon', 'stop_lat')

## Spatial Join (which points fall into which polygon)

This URL gives you CA county boundaries: https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12

* Go to "I want to use this" > View API Resources > copy link for geojson 
* [Link to Geojson](https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson)
* Read in the geojson with `geopandas` and make it a geodataframe: `gpd.read_file(LONG_URL_PATH)`
* Double check that the coordinate reference system is the same for both gdfs using `gdf.crs`. If not, change it so they are the same.
* Spatial join stops to counties: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html)
    * Play with inner join or left join, what's the difference? Which one do you want?
    * Play with switching around the left_df and right_df, what's the right order?


In [ ]:
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson')

#### Checking to see if the 2 dataframes have the same coordinate system, WGS 84 

In [ ]:
gdf.crs 

In [ ]:
geojson.crs

#### Spatial Join

* Left join: with Cal ITP data on the left is the same as an inner join between Cal ITP & county boundaries data frame. 

* The left join with county data on the left was totally crazy? There were many more rows. I guess some rows from the Cal ITP data didn't match up with any of the counties.

In [ ]:
#inner
spatial_inner_data = gpd.sjoin(gdf, geojson, how='inner')

spatial_inner_data.shape

In [ ]:
#left join 1
spatial_left1 = gpd.sjoin(gdf, geojson, how='left')

spatial_left1.shape

In [ ]:
#checking to see if inner and first left join data frame are the same..false. I checked in the spreadsheets and realized the
#rows were all jumbled 
spatial_left1.equals(spatial_inner_data)

In [ ]:
#left join 2 switching data frames
spatial_left2 = gpd.sjoin(geojson,gdf, how='left')


In [ ]:
#this has way more columns..looks totally inaccurate
spatial_left2.shape

## By county: count number of stops and stops per sq_mi
* By county: count number of stops and stops per sq_mi.
    * Hint 1: Start with a CRS with units in feet or meters, then do a conversion to sq mi. [CRS in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py)
    * Hint 2: to find area, you can create a new column and calculate `gdf.geometry.area`. [geometry manipulations docs](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html)
    
<b>  Notes from 1/7/2021 </b>
* Several California CRS some are in feet some are in meter
* Lat and lon will be stored as Wgs84, units are decimal meters
* Most of the time you have to convert
* Want main data frame on the left, but when you map it have to replace geometry column of bus points to the county geometry
* Left/Inner and aggregate on county level, count how many stops, merge in shape file
* By county: count number of stops and stops per sq_mi.
    * Hint 1: Start with a CRS with units in feet or meters, then do a conversion to sq mi. [CRS in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py)
    * Hint 2: to find area, you can create a new column and calculate `gdf.geometry.area`. [geometry manipulations docs](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html)
   

#### Stops per County first

In [ ]:
#changing data type to 2229 
spatial_inner_data = spatial_inner_data.to_crs(epsg =2229)

In [ ]:
#subsetting for the columns of interest.
subset = spatial_inner_data[['geometry','COUNTY_NAME','stop_id']]

In [ ]:
subset.head(2)

In [ ]:
#aggregating stops by counting unique stops per county.
bus_stops_county = subset.dissolve(by='COUNTY_NAME', aggfunc ='nunique').reset_index()

In [ ]:
#have to rename bc I keep getting an error when merging
bus_stops_county = bus_stops_county.rename(columns={'index_left': 'left', 'index_right':'right'})

In [ ]:
#converting geojson to correct crs of bus stops.
geojson_epsg229 = geojson.to_crs(bus_stops_county.crs)

In [ ]:
#merge in geojson with stops by county..
bus_stops_joined = gpd.sjoin(bus_stops_county, geojson_epsg229, how='left')

In [ ]:
#drop columns I don't need
bus_stops_joined = bus_stops_joined[['geometry','COUNTY_NAME_left','stop_id']]

In [ ]:
#rename columns
bus_stops_joined = bus_stops_joined.rename(columns = {'stop_id': 'Count_of_Stops', 'COUNTY_NAME_left': 'County_Name'})

In [ ]:
#looking at dataframe
bus_stops_joined 

In [ ]:
#just making sure my results above are  right, I am surprised to see Merced County has the most stops
testing = spatial_inner_data.groupby(['COUNTY_NAME']).agg({'stop_id':'nunique'}).reset_index()
testing

#### Stops per Square Mile 

Tiffany's Feedback
* EPSG:2229 is in ft. geometry.area gives units in sq ft.
* Use a different conversion to turn sq ft into sq mi.

Amanda
* [Apparently there are 27878400 square feet in 1 square mile?](https://www.unitconverters.net/area/square-foot-to-square-mile.htm) I'm having trouble finding a reliable resource...
* Checking just to see square miles on the wikipedia pages of Alameda, Contra Costa, and Santa Clara, mine looks somewhat correct? But not completely...
* Alameda is supposed to be 739 sq miles per wikipediaContra Costa is 716...



In [ ]:
bus_stops_joined['Square_Miles'] = bus_stops_joined['geometry'].area / 27878400

In [ ]:
bus_stops_joined

In [ ]:
#dividing number of stops by square miles
bus_stops_joined["Stops_per_Sq"] = bus_stops_joined['Count_of_Stops'] / bus_stops_joined['Square_Miles']

In [ ]:
#dropping columns 
bus_stops_sqmile = bus_stops_joined[['geometry','County_Name','Stops_per_Sq']]


In [ ]:
bus_stops_sqmile.round(2)